# 02 - Data Cleaning and Master Dataset

Phase 3 (cleaning) and Phase 4 (master dataset) of the project.

Applies the documented, reproducible cleaning pipeline from
`src/data_processing/cleaning.py` to every raw category file, validates the
result, builds the unified Master Dataset (`src/data_processing/master_dataset.py`),
cross-checks it against the user's own prior master-dataset attempt, and saves
everything under `data/processed/`.

Raw files under `data/raw/` are never modified -- this notebook only reads
them and writes new files under `data/processed/`.

## 1. Setup

In [1]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src").is_dir() and (parent / "data").is_dir():
            return parent
    raise RuntimeError("Could not locate project root from " + str(start))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

PROJECT_ROOT

WindowsPath('F:/Instant/DataAnalysis/Gradution Project/All data/_staging')

In [2]:
from src.config import RAW_CATEGORY_FILES, PROCESSED_DATA_DIR, PREVIOUS_WORK_DIR, REPORTS_DIR
from src.data_processing.load_raw import load_all_raw
from src.data_processing.cleaning import clean_all, flag_price_outliers
from src.data_processing.master_dataset import build_master_dataset

## 2. Load Raw Data

In [3]:
raw = load_all_raw()
{k: v.shape for k, v in raw.items()}

{'ceilings_gypsum.csv': (260, 22),
 'doors.csv': (1781, 21),
 'electrical_basic.csv': (938, 19),
 'electrical_finishing.csv': (555, 19),
 'flooring.csv': (1011, 19),
 'paints.csv': (165, 23),
 'plumbing_basic.csv': (229, 19),
 'sanitary_ware.csv': (1669, 19),
 'labor_and_auxiliary.csv': (9, 13)}

## 3. Clean Every Category (Phase 3)

Cleaning steps applied (see `src/data_processing/cleaning.py` for the full,
documented implementation):

1. Strip whitespace from every text column.
2. Standardise the one vocabulary inconsistency found in the audit
   (`Required_For`: `"Bedrooms"` -> `"Bedroom"`).
3. Coerce `Price_EGP`, `Rule_Value`, `Quality_Score` to numeric.
4. Drop exact duplicate rows.
5. Defensively drop any row with a missing/non-positive price (none exist in
   the current snapshot -- this guards future data refreshes).

No prices are capped or winsorized: the audit (notebook 01) found the flagged
outliers to be legitimate premium products (e.g. large security doors,
designer sanitary ware), not data errors, so they are kept as-is with an
informational `Price_Outlier` flag only.

In [4]:
cleaned = clean_all(raw)

cleaning_log = pd.DataFrame({
    "Rows_Before": {k: len(v) for k, v in raw.items()},
    "Duplicates_Removed": {k: v.attrs.get("n_duplicates_removed", 0) for k, v in cleaned.items()},
    "Invalid_Price_Removed": {k: v.attrs.get("n_invalid_price_removed", 0) for k, v in cleaned.items()},
    "Rows_After": {k: len(v) for k, v in cleaned.items()},
})
cleaning_log.loc["TOTAL"] = cleaning_log.sum()
cleaning_log

,Rows_Before,Duplicates_Removed,Invalid_Price_Removed,Rows_After
ceilings_gypsum.csv,260,11,0,249
doors.csv,1781,26,0,1755
electrical_basic.csv,938,47,0,891
electrical_finishing.csv,555,16,0,539
flooring.csv,1011,7,0,1004
paints.csv,165,15,0,150
plumbing_basic.csv,229,0,0,229
sanitary_ware.csv,1669,8,0,1661
labor_and_auxiliary.csv,9,0,0,9
TOTAL,6617,130,0,6487


In [5]:
# Verify the vocabulary fix took effect.
fixed = cleaned["flooring.csv"]["Required_For"].value_counts()
assert "Bedrooms" not in fixed.index, "Bedrooms -> Bedroom fix did not apply"
fixed

Required_For
Reception    631
Bedroom      373
Name: count, dtype: Int64

## 4. Flag Price Outliers (informational only)

In [6]:
for filename, df in cleaned.items():
    cleaned[filename] = flag_price_outliers(df)

outlier_summary = pd.Series(
    {k: int(v["Price_Outlier"].sum()) for k, v in cleaned.items()},
    name="Flagged_Outliers_Kept",
).to_frame()
outlier_summary

,Flagged_Outliers_Kept
ceilings_gypsum.csv,0
doors.csv,23
electrical_basic.csv,70
electrical_finishing.csv,9
flooring.csv,0
paints.csv,0
plumbing_basic.csv,5
sanitary_ware.csv,0
labor_and_auxiliary.csv,0


## 5. Save Cleaned Per-Category Tables

Each cleaned category table is saved individually (preserving every
category-specific column) before being combined into the master dataset.

In [7]:
for filename, df in cleaned.items():
    out_path = PROCESSED_DATA_DIR / filename
    df.to_csv(out_path, index=False)
    print("Saved", out_path, df.shape)

Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\ceilings_gypsum.csv (249, 23)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\doors.csv (1755, 22)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\electrical_basic.csv (891, 20)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\electrical_finishing.csv (539, 20)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\flooring.csv (1004, 20)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\paints.csv (150, 24)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\plumbing_basic.csv (229, 20)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\sanitary_ware.csv (1661, 20)
Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\labor_and_auxiliary.csv (9, 14)


## 6. Build the Master Dataset (Phase 4)

Categories are concatenated with an outer column union: a category-specific
column (e.g. `Power_W`, `Coverage_m2_per_L`, `Size_cm`) is simply `NaN` for
rows from categories where it does not apply, rather than being dropped.
A stable `Product_ID` (`P000001`, `P000002`, ...) is assigned across the
whole master table.

In [8]:
master = build_master_dataset(cleaned)
print(master.shape)
master.head()

(6487, 39)


,Product_ID,Finishing_Category,Category,Subcategory,Product_Name,Brand,Material,Product_Type,Quality_Level,Quality_Score,Grade,Quality_Price_Band,Price_EGP,Unit,Quantity_Rule,Rule_Value,Required_For,Optional,Application,Specification,Ceiling_Type,Coats,Component,Coverage_m2,Coverage_m2_per_L,Door_Category,Door_Type,Electrical_Category,Finish,Floor_Type,Paint_Category,Paint_Type,Plumbing_Category,Power_W,Price_Outlier,Size,Size_cm,Source_File,Waste_Factor
0,P000001,Ceilings,Ceilings,Decorative Gypsum Ceiling,Decorative Gypsum Ceiling,Knauf,Gypsum,<NA>,Low,8.4,Not Applicable,Low: <= 234,203,Piece,Area_m2/Coverage,1.0,Reception,True,Reception,60x60 cm,Decorative Gypsum Ceiling,NaN,Decorative Panel,0.36,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,<NA>,ceilings_gypsum.csv,NaN
1,P000002,Ceilings,Ceilings,Decorative Gypsum Ceiling,Decorative Gypsum Ceiling,USG Boral,Gypsum,<NA>,Low,8.4,Not Applicable,Low: <= 234,219,Piece,Area_m2/Coverage,1.0,Reception,True,Reception,60x60 cm,Decorative Gypsum Ceiling,NaN,Decorative Panel,0.36,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,<NA>,ceilings_gypsum.csv,NaN
2,P000003,Ceilings,Ceilings,Gypsum Board Ceiling,Gypsum Board Ceiling,Gyproc,Gypsum,<NA>,Low,8.7,Not Applicable,Low: <= 234,231,Sheet,Area_m2/Coverage,1.0,Ceiling,False,Full Ceiling,12.5 mm / 120x240 cm,Gypsum Board Ceiling,NaN,Gypsum Board,2.88,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,<NA>,ceilings_gypsum.csv,NaN
3,P000004,Ceilings,Ceilings,Decorative Gypsum Ceiling,Decorative Gypsum Ceiling,Gyproc,Gypsum,<NA>,Medium,8.4,Not Applicable,Medium: 234 - 285,281,Piece,Area_m2/Coverage,1.0,Reception,True,Reception,60x60 cm,Decorative Gypsum Ceiling,NaN,Decorative Panel,0.36,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,<NA>,ceilings_gypsum.csv,NaN
4,P000005,Ceilings,Ceilings,Gypsum Board Ceiling,Gypsum Board Ceiling,Siniat,Gypsum,<NA>,High,8.7,Not Applicable,High: > 285,298,Sheet,Area_m2/Coverage,1.0,Ceiling,False,Full Ceiling,12.5 mm / 120x240 cm,Gypsum Board Ceiling,NaN,Gypsum Board,2.88,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,False,<NA>,<NA>,ceilings_gypsum.csv,NaN


## 7. Validate the Master Dataset

In [9]:
checks = {
    "Product_ID is unique": master["Product_ID"].is_unique,
    "No missing Price_EGP": master["Price_EGP"].notna().all(),
    "No non-positive Price_EGP": (master["Price_EGP"] > 0).all(),
    "No fully-duplicate rows (excl. Product_ID)": not master.drop(columns=["Product_ID"]).duplicated().any(),
    "Every category present": set(master["Finishing_Category"].unique()) == set(RAW_CATEGORY_FILES.values()),
    "Row count matches sum of cleaned categories": len(master) == sum(len(v) for v in cleaned.values()),
}
validation_table = pd.DataFrame({"Check": checks.keys(), "Passed": checks.values()})
assert validation_table["Passed"].all(), "Master dataset failed validation"
validation_table

,Check,Passed
0,Product_ID is unique,True
1,No missing Price_EGP,True
2,No non-positive Price_EGP,True
3,No fully-duplicate rows (excl. Product_ID),True
4,Every category present,True
5,Row count matches sum of cleaned categories,True


In [10]:
category_counts = master["Finishing_Category"].value_counts()
category_counts

Finishing_Category
Doors                   1755
Sanitary Ware           1661
Flooring                1004
Electrical Basic         891
Electrical Finishing     539
Ceilings                 249
Plumbing Basic           229
Paints                   150
Labor and Auxiliary        9
Name: count, dtype: Int64

## 8. Cross-Check Against Prior Master-Dataset Attempt

The user's own earlier work already produced a first master-dataset attempt
(`data/previous_work/master_finishing_products.csv`, preserved untouched).
As a sanity check, this project's independently rebuilt master dataset is
compared against it -- close agreement in category composition validates
that the raw-file selection and category labeling in Section 3 of notebook
01 was correct.

In [11]:
previous_master = pd.read_csv(PREVIOUS_WORK_DIR / "master_finishing_products.csv")

comparison = pd.DataFrame({
    "This_Project_Master": category_counts,
    "Previous_Work_Master": previous_master["Finishing_Category"].value_counts(),
}).fillna(0).astype(int)
comparison["Difference"] = comparison["This_Project_Master"] - comparison["Previous_Work_Master"]
comparison

,This_Project_Master,Previous_Work_Master,Difference
Finishing_Category,,,
Ceilings,249,249,0
Doors,1755,1755,0
Electrical Basic,891,891,0
Electrical Finishing,539,539,0
Flooring,1004,1004,0
Labor and Auxiliary,9,9,0
Paints,150,150,0
Plumbing Basic,229,229,0
Sanitary Ware,1661,1661,0


The difference is **zero for every category**. This confirms the Section 3
finding precisely: the previous master-dataset attempt was built from the
same already-deduplicated "model-ready" files that were verified to be
sitting in `All data/` (not from the true raw files, which still contained
the 130 exact-duplicate rows removed in Section 3). This project's master
dataset therefore reproduces the previous attempt's product coverage exactly
while additionally documenting *why* each row count is what it is, tracing
back to true raw sources, and keeping every category-specific column instead
of only the shared core schema.

In [12]:
assert (comparison["Difference"] == 0).all(), "Expected exact agreement with the previous master dataset"
print("Category composition matches the previous master-dataset attempt exactly.")

Category composition matches the previous master-dataset attempt exactly.


## 9. Save the Master Dataset

In [13]:
master_path = PROCESSED_DATA_DIR / "master_products.csv"
master.to_csv(master_path, index=False)
print("Saved", master_path, master.shape)

Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\data\processed\master_products.csv (6487, 39)


In [14]:
report_path = REPORTS_DIR / "02_cleaning_and_master_dataset_report.md"
lines = ["# Data Cleaning & Master Dataset Report", ""]
lines.append("Generated by `notebooks/02_Data_Cleaning_and_Master_Dataset.ipynb`. Do not edit by hand.")
lines.append("")
lines.append("## Cleaning log")
lines.append("")
lines.append(cleaning_log.to_markdown())
lines.append("")
lines.append("## Master dataset validation")
lines.append("")
lines.append(validation_table.to_markdown(index=False))
lines.append("")
lines.append("## Category composition vs. previous master-dataset attempt")
lines.append("")
lines.append(comparison.to_markdown())
lines.append("")
lines.append(f"Final master dataset: **{master.shape[0]} rows x {master.shape[1]} columns**, saved to `data/processed/master_products.csv`.")

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Saved", report_path)

Saved F:\Instant\DataAnalysis\Gradution Project\All data\_staging\reports\02_cleaning_and_master_dataset_report.md
